# Lower Volta Final Analysis
Run the Earth Engine scripts first, start all required exports, then execute this notebook from top to bottom.


## 01_install_and_config.py


In [ ]:
# 01 — Environment and configuration
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports')
except Exception:
    ROOT = Path('data/exports')

OUT = Path('outputs')
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'study_area_km2': 8938.927,
    'event_temporary_km2': 5.868,
    'cumulative_km2': 7.386,
    'recurrent_km2': 4.487,
    'sporadic_km2': 2.899,
    'surface_change_km2': 2032.146,
    'TP': 278, 'TN': 297, 'FP': 3, 'FN': 22
}
print('Input:', ROOT)
print('Output:', OUT)


## 02_threshold_benchmark.py


In [ ]:
# 02 — Diagnostic method benchmark and threshold sensitivity figure
#
# The manuscript's F1 benchmark and 3x3 F1 matrix are archived outputs from the
# diagnostic comparison analysis. They are intentionally read from versioned
# repository CSV files instead of being silently recreated from a new random
# sample. GEE script 05 independently recomputes Otsu thresholds, mapped areas,
# and the threshold prediction stack from the satellite imagery.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

# Resolve repository root whether the script is run from repo root or colab/.
HERE = Path.cwd()
if (HERE / 'data' / 'archived').exists():
    REPO = HERE
elif (HERE.parent / 'data' / 'archived').exists():
    REPO = HERE.parent
else:
    raise FileNotFoundError('Could not locate data/archived in the repository.')

benchmark_path = REPO / 'data' / 'archived' / 'LV_Diagnostic_Benchmark_Metrics.csv'
sensitivity_path = REPO / 'data' / 'archived' / 'LV_S1_Threshold_F1_Sensitivity.csv'

benchmark = pd.read_csv(benchmark_path)
sensitivity = pd.read_csv(sensitivity_path)

benchmark.to_csv(OUT / 'Table3_method_benchmark_archived.csv', index=False)
sensitivity.to_csv(OUT / 'Supplementary_threshold_sensitivity_archived.csv', index=False)

# Figure 5a
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(benchmark['method'], benchmark['f1_pct'])
ax.set_ylabel('F1-score (%)')
ax.set_ylim(50, 100)
ax.set_title('(a)', loc='left', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.25)
for bar, value in zip(bars, benchmark['f1_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, value + 1, f'{value:.1f}',
            ha='center', fontweight='bold')
fig.tight_layout()
fig.savefig(OUT / 'Figure5a_method_benchmark.png', dpi=600, bbox_inches='tight')
plt.close(fig)

# Figure 5b
matrix = sensitivity.set_index('vh_threshold_db')[['vv_-18', 'vv_-17', 'vv_-16']]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix.values, vmin=60, vmax=80, aspect='auto')
ax.set_xticks(range(3), ['-18', '-17', '-16'])
ax.set_yticks(range(3), [str(int(x)) for x in matrix.index])
ax.set_xlabel('VV threshold (dB)')
ax.set_ylabel('VH threshold (dB)')
ax.set_title('(b)', loc='left', fontweight='bold')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f'{matrix.iloc[i, j]:.1f}', ha='center', va='center',
                fontweight='bold')
cb = fig.colorbar(im, ax=ax)
cb.set_label('F1-score (%)')
fig.tight_layout()
fig.savefig(OUT / 'Figure5b_threshold_sensitivity.png', dpi=600, bbox_inches='tight')
plt.close(fig)

print('Archived diagnostic method benchmark:')
print(benchmark.to_string(index=False))
print('\nArchived F1 sensitivity matrix:')
print(matrix)


## 03_hydroclimate_stats.py


In [ ]:
# 03 — Hydro-climatic correlations
from pathlib import Path
import pandas as pd
from scipy.stats import pearsonr, spearmanr

ROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
p=ROOT/'LV_Hydroclimate_Per_Observation.csv'
if not p.exists():
    raise FileNotFoundError(f'Missing {p}; run GEE script 03 first.')
df=pd.read_csv(p)
rows=[]
for prefix,label in [('rain_','CHIRPS rainfall'),('runoff_','ERA5-Land runoff')]:
    for d in [1,3,7,14,30]:
        c=f'{prefix}{d}d_mm'
        if c not in df.columns: continue
        x=df[['temporary_flood_km2',c]].dropna()
        if len(x)<3: continue
        pr,pp=pearsonr(x['temporary_flood_km2'],x[c])
        sr,sp=spearmanr(x['temporary_flood_km2'],x[c])
        rows.append({'variable':label,'window_days':d,'n':len(x),
                     'pearson_r':pr,'pearson_p':pp,'spearman_rho':sr,'spearman_p':sp})
corr=pd.DataFrame(rows)
corr.to_csv(OUT/'Supplementary_Table_S3_hydroclimatic_correlations.csv',index=False)
print(corr.to_string(index=False))


## 04_agreement_metrics.py


In [ ]:
# 04 — Sentinel-1 / Sentinel-2 inter-sensor agreement
from pathlib import Path
import pandas as pd
from sklearn.metrics import confusion_matrix

ROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
p=ROOT/'LV_S1_S2_600_Point_Comparison.csv'
if not p.exists():
    raise FileNotFoundError(f'Missing {p}. Run corrected GEE script 03 and export the actual 600-point comparison CSV. Archived manuscript counts are intentionally NOT substituted.')
d=pd.read_csv(p)
required={'s2_class','s1_class'}
if not required.issubset(d.columns): raise ValueError(f'{p} must contain {sorted(required)}')
d=d.dropna(subset=['s2_class','s1_class'])
if len(d)!=600: raise ValueError(f'Expected 600 valid comparison samples; found {len(d)}.')
tn,fp,fn,tp=confusion_matrix(d['s2_class'].astype(int),d['s1_class'].astype(int),labels=[0,1]).ravel()
n=tp+tn+fp+fn; oa=(tp+tn)/n; precision=tp/(tp+fp); recall=tp/(tp+fn); specificity=tn/(tn+fp)
f1=2*precision*recall/(precision+recall); omission=fn/(tp+fn); commission=fp/(tp+fp)
nonf_prod=tn/(tn+fp); nonf_user=tn/(tn+fn); nonf_omission=fp/(tn+fp); nonf_commission=fn/(tn+fn)
pe=((tp+fn)/n)*((tp+fp)/n)+((tn+fp)/n)*((tn+fn)/n); kappa=(oa-pe)/(1-pe)
metrics=pd.DataFrame({'metric':['TP','TN','FP','FN','Total samples','Overall agreement (%)','Flood precision (%)','Flood recall (%)','Flood specificity (%)','Flood F1-score (%)','Flood omission error (%)','Flood commission error (%)','Non-flood producer agreement (%)','Non-flood user agreement (%)','Non-flood omission error (%)','Non-flood commission error (%)','Cohen kappa'],'value':[tp,tn,fp,fn,n,oa*100,precision*100,recall*100,specificity*100,f1*100,omission*100,commission*100,nonf_prod*100,nonf_user*100,nonf_omission*100,nonf_commission*100,kappa]})
metrics.to_csv(OUT/'Table5_inter_sensor_metrics.csv',index=False); print(metrics.to_string(index=False))


## 05_tables_figures.py


In [ ]:
# 05 — Consolidate tables from Earth Engine exports
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Lower_Volta_Final_Exports') if Path('/content/drive').exists() else Path('data/exports')
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)

def load(name):
    p=ROOT/name
    if not p.exists():
        print('Missing:',p); return None
    return pd.read_csv(p)

core=load('LV_Core_Area_Summary.csv')
terrain=load('LV_Terrain_Statistics.csv')
wc=load('LV_WorldCover_Exposure.csv')
ghsl=load('LV_GHSL_Built_Exposure.csv')
mndwi=load('LV_MNDWI_Sensitivity.csv')
monthly=load('LV_CHIRPS_Monthly_2023.csv')

if core is not None: core.to_csv(OUT/'Table2_core_surface_water_results.csv',index=False)
if terrain is not None: terrain.to_csv(OUT/'Supplementary_Table_S4_terrain_statistics.csv',index=False)
if wc is not None:
    c=wc[wc['zone'].eq('Cumulative temporary flood')].copy()
    c['share_pct']=c['area_km2']/c['area_km2'].sum()*100
    c.sort_values('area_km2',ascending=False).to_csv(OUT/'Supplementary_Table_S5_WorldCover.csv',index=False)
if ghsl is not None:
    ghsl.to_csv(OUT/'GHSL_built_exposure_summary.csv',index=False)
if mndwi is not None: mndwi.to_csv(OUT/'Supplementary_Table_S6_MNDWI_sensitivity.csv',index=False)
if monthly is not None: monthly.sort_values('month').to_csv(OUT/'CHIRPS_monthly_2023.csv',index=False)

pd.DataFrame({
 'quantity':['Study area','Event temporary flood','Cumulative temporary flood','Recurrent temporary flood','Sporadic component','Surface-change zone'],
 'expected_km2':[8938.927,5.868,7.386,4.487,2.899,2032.146]
}).to_csv(OUT/'manuscript_expected_values.csv',index=False)
print('Final tables saved.')
